In [ ]:
from huggingface_hub import login

# Authenticate with Hugging Face
login("HF_TOKEN")

In [ ]:
# @title
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

# ================= CẤU HÌNH COPY =================
# Đường dẫn gốc trên Drive (Source)
SOURCE_DIR = "/content/drive/MyDrive/Thien/splitthienfinal"

# Đường dẫn đích trên Colab (Destination)
DEST_DIR = "/content/chatterbox-lora/audio_data"

# Số luồng copy cùng lúc (Colab chịu được khoảng 8-16 luồng ổn định)
NUM_THREADS = 16

def copy_file(src, dst):
    try:
        shutil.copy2(src, dst)
    except Exception as e:
        print(f"Lỗi copy {src}: {e}")

def copy_folder_multithreaded(source_folder, dest_folder):
    # Tạo danh sách tất cả các file cần copy
    files_to_copy = []
    print(f"🔍 Đang quét file từ: {source_folder} ...")

    for root, dirs, files in os.walk(source_folder):
        # Tạo cấu trúc thư mục tương ứng ở đích
        rel_path = os.path.relpath(root, source_folder)
        dest_path = os.path.join(dest_folder, rel_path)
        os.makedirs(dest_path, exist_ok=True)

        for file in files:
            src_file = os.path.join(root, file)
            dst_file = os.path.join(dest_path, file)
            # Chỉ copy nếu file chưa tồn tại
            if not os.path.exists(dst_file):
                files_to_copy.append((src_file, dst_file))

    print(f"📦 Tìm thấy {len(files_to_copy)} file cần copy.")
    print(f"🚀 Bắt đầu copy đa luồng sang {dest_folder}...")

    # Thực hiện copy đa luồng
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        list(tqdm(
            executor.map(lambda p: copy_file(p[0], p[1]), files_to_copy),
            total=len(files_to_copy),
            unit='file'
        ))

    print("✅ Copy hoàn tất!")

# Thực thi
if __name__ == "__main__":
    if os.path.exists(SOURCE_DIR):
        copy_folder_multithreaded(SOURCE_DIR, DEST_DIR)
    else:
        print(f"❌ Không tìm thấy thư mục nguồn: {SOURCE_DIR}")

In [ ]:
# @title
# 4. Tải toàn bộ model về Drive (snapshot_download sẽ download tất cả files: weights, config, processor files...)
from huggingface_hub import snapshot_download

save_path = "/content/drive/MyDrive/AI_Models/sam-audio-large"  # Thay đường dẫn nếu muốn

snapshot_download(
    repo_id="facebook/sam-audio-large",  # Hoặc -base/-small nếu muốn nhẹ hơn
    local_dir=save_path,
    local_dir_use_symlinks=False,  # Để copy thật vào Drive, không link
    resume_download=True  # Nếu đứt quãng thì tiếp tục
)

print(f"Model đã tải xong toàn bộ files và lưu tại: {save_path}")
print("Lần sau chỉ mount Drive là dùng được weights offline!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

checkpoint.pt:   0%|          | 0.00/14.9G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.37k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

LICENSE:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

Model đã tải xong toàn bộ files và lưu tại: /content/drive/MyDrive/AI_Models/sam-audio-large
Lần sau chỉ mount Drive là dùng được weights offline!


In [ ]:
# Clone repo
!git clone https://github.com/facebookresearch/sam-audio.git
%cd sam-audio

Cloning into 'sam-audio'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 197 (delta 78), reused 55 (delta 55), pack-reused 90 (from 1)
Receiving objects: 100% (197/197), 17.03 MiB | 24.60 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/sam-audio


In [ ]:
# Cài đặt package (editable mode, sẽ tự pull dependencies như torch, torchaudio, transformers, v.v.)
!pip install -e .

Obtaining file:///content/sam-audio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/facebookresearch/dacvae.git to /tmp/pip-install-y8mcdyvp/dacvae_0f55f8975b6d4ae99c7c741d8a575a9a
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/dacvae.git /tmp/pip-install-y8mcdyvp/dacvae_0f55f8975b6d4ae99c7c741d8a575a9a
  Resolved https://github.com/facebookresearch/dacvae.git to commit 414c20785fc3a28373073ea8ef7a1316eeeaca6e
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/ImageBind.git to /tmp/pip-install-y8mcdyvp/imagebind_cf01da752bab441a9605d6f79818294d
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/ImageBind.git /tmp/pip-install-y8mcdyvp/imagebind_cf01da752bab441a9605d6f79818294d
  Reso

In [ ]:
# Gỡ cài đặt TensorFlow và các package liên quan gây conflict
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-io-gcp tensorflow-text tensorflow-hub tensorflow-datasets tensorflow-metadata tensorflow-probability tb-nightly

# Cài đặt transformers mới nhất (để có ModernBertConfig) NHƯNG giữ huggingface_hub < 0.25.0
# vì HF Hub 0.25.0+ đã bỏ tham số proxies và resume_download gây lỗi cho SAMAudio
!pip install "transformers>=4.48.0" "huggingface_hub<0.25.0" "protobuf==3.20.3"

# Khởi động lại kernel tự động để Python load thư viện mới nhất vào bộ nhớ
import os
print("🔄 Đang tự động Restart Runtime để áp dụng thay đổi...")
os.kill(os.getpid(), 9)


Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: tensorflow-text 2.19.0
Uninstalling tensorflow-text-2.19.0:
  Successfully uninstalled tensorflow-text-2.19.0
Found existing installation: tensorflow-hub 0.16.1
Uninstalling tensorflow-hub-0.16.1:
  Successfully uninstalled tensorflow-hub-0.16.1
Found existing installation: tensorflow-datasets 4.9.9
Uninstalling tensorflow-datasets-4.9.9:
  Successfully uninstalled tensorflow-datasets-4.9.9
Found existing installation: tensorflow-metadata 1.17.3
Uninstalling tensorflow-metadata-1.17.3:
  Successfully uninstalled tensorflow-metadata-1.17.3
Found existing installation: tensorflow-probability 0.25.0
Uninstalling tensorflow-probability-0.25.0:
  Successfully uninstalled tensorflow-probability-0.25.0
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a 

KeyboardInterrupt: 

# Load local

In [ ]:
# @title
import torch
from huggingface_hub import login
from sam_audio import SAMAudio, SAMAudioProcessor
import torchaudio


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load model (lần đầu sẽ download ~ vài GB, sau đó cache)
model = SAMAudio.from_pretrained("/content/drive/MyDrive/AI_Models/sam-audio-large").to(device).eval()
processor = SAMAudioProcessor.from_pretrained("/content/drive/MyDrive/AI_Models/sam-audio-large")



In [ ]:
import torch
from huggingface_hub import login
from sam_audio import SAMAudio, SAMAudioProcessor
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load model directly from Hugging Face
repo_id = "facebook/sam-audio-large" # Or -base/-small if you prefer

model = SAMAudio.from_pretrained(repo_id).to(device).eval()
processor = SAMAudioProcessor.from_pretrained(repo_id)


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

# Run

In [ ]:
# Define the audio file and description centrally
audio_file = "/content/X7cPLobvtnw.wav" # Ensure this is the correct path to your audio file
description = "woman speaking"  # Mô tả âm thanh muốn tách

In [ ]:
anchors = [
    ["+", 0, 5.0],  # Sound occurs between 6.3 and 7.0 seconds
]

inputs = processor(audios=[audio_file], descriptions=[description], anchors=[anchors]).to(device)

with torch.inference_mode():
    result = model.separate(inputs, predict_spans=True, reranking_candidates=6)

# Lưu kết quả
torchaudio.save("target.wav", result.target[0].cpu(), processor.audio_sampling_rate)
torchaudio.save("residual.wav", result.residual[0].cpu(), processor.audio_sampling_rate)

# @title
import matplotlib.pyplot as plt
import torchaudio
import numpy as np
from IPython.display import Audio, display

def show_audio_and_waveform(file_path, title):
    print(f"\n--- {title} ---")
    # Load audio để vẽ biểu đồ
    waveform, sr = torchaudio.load(file_path)

    # Lấy channel đầu tiên và chuyển về numpy array để plot
    audio_data = waveform[0].numpy()
    time_axis = np.arange(len(audio_data)) / sr

    # Vẽ biểu đồ dạng sóng
    plt.figure(figsize=(12, 3))
    plt.plot(time_axis, audio_data, color="dodgerblue", alpha=0.8)
    plt.title(title)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

    # Hiển thị audio player
    display(Audio(file_path, autoplay=False))

# Hiển thị lần lượt 3 file
show_audio_and_waveform(audio_file, "Original Audio")
show_audio_and_waveform("target.wav", "Target Audio (Separated)")
show_audio_and_waveform("residual.wav", "Residual Audio (Leftover)")

